# 02 — Revenue, demand and inventory forecasting

This notebook follows the production forecasting path: candidate models are compared on a chronological validation window, the validation winner is selected, and the untouched holdout is reported separately. Inventory calculations reuse the product demand forecasts and explicit business assumptions.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "backend").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "backend"))

import pandas as pd

from app.database.base import SessionLocal, init_db
from app.services.data_service import ensure_demo_dataset, get_context

init_db()
db = SessionLocal()
dataset = ensure_demo_dataset(db)
ctx = get_context(dataset.id, db)
print(f"Forecasting through {ctx.date_max} with {ctx.capabilities}")

## Revenue forecast

The candidate table comes from validation only. The deployed model is the candidate with the lowest validation MAE; holdout metrics are shown after selection.

In [ ]:
from app.services.revenue_service import revenue_analysis, revenue_forecast

analysis = revenue_analysis(ctx, granularity="month")
forecast = revenue_forecast(ctx, horizon_days=90)
pd.Series(analysis["kpis"], name="value")

In [ ]:
summary = forecast["summary"]
pd.Series(
    {
        "model": summary["model_label"],
        "trend": summary["trend"],
        "validation_quality": summary["validation_quality"],
        "forecast_7d": summary["forecast_7d"],
        "forecast_30d": summary["forecast_30d"],
        "forecast_90d": summary["forecast_90d"],
        "recent_30d_actual": summary["recent_30d_actual"],
        "growth_30_pct": summary["growth_30_pct"],
        "holdout_mae": summary["metrics"].get("mae"),
        "holdout_mape": summary["metrics"].get("mape"),
    },
    name="value",
)

In [ ]:
candidates = pd.json_normalize(summary["candidates"])
candidates[["label", "model", "status", "metrics.mae", "metrics.rmse", "metrics.mape"]]

In [ ]:
validation = pd.DataFrame(forecast["validation"])
validation

## Product demand forecasts

Products are ranked by recent revenue before forecasting, then by forecast demand. Products with insufficient history remain visible and are explicitly marked unavailable instead of receiving a fabricated forecast.

In [ ]:
from app.services.product_service import forecast_products

product_result = forecast_products(ctx, limit=12)
pd.Series(product_result["summary"], name="value")

In [ ]:
products = pd.DataFrame(product_result["products"])
products[
    [
        "rank",
        "product",
        "forecast_30d",
        "forecast_30_growth_pct",
        "model",
        "validation_quality",
        "potential",
        "trend",
        "status",
    ]
].head(12)

## Inventory plan

Safety stock, reorder point and EOQ are planning calculations, not guaranteed savings. Change the assumptions to see their effect.

In [ ]:
from app.services.inventory_service import inventory_analysis

inventory = inventory_analysis(
    ctx,
    lead_time_days=7,
    service_level=0.95,
    ordering_cost=500.0,
    holding_rate=0.20,
)
pd.Series(inventory["assumptions"], name="value")

In [ ]:
pd.Series(inventory["summary"], name="value")

In [ ]:
items = pd.DataFrame(inventory["items"])
items[
    ["product", "status", "current_stock", "forecast_30d", "coverage_days", "reorder_point", "suggested_order_qty", "eoq"]
].head(15)

In [ ]:
recommendations = pd.DataFrame(inventory["recommendations"])
recommendations[["product", "status", "priority", "issue", "actions"]].head(10)

In [ ]:
db.close()